In [1]:
import pandas as pd
import numpy as np

dataset_cont = pd.read_csv('C:/Users/tf/Desktop/airbnb_open_data.csv')
print(dataset_cont)

             id                                              NAME  \
0       1001254                Clean & quiet apt home by the park   
1       1002102                             Skylit Midtown Castle   
2       1002403               THE VILLAGE OF HARLEM....NEW YORK !   
3       1002755                                               NaN   
4       1003689  Entire Apt: Spacious Studio/Loft by central park   
...         ...                                               ...   
102594  6092437                        Spare room in Williamsburg   
102595  6092990                     Best Location near Columbia U   
102596  6093542                    Comfy, bright room in Brooklyn   
102597  6094094                  Big Studio-One Stop from Midtown   
102598  6094647                              585 sf Luxury Studio   

            host id host_identity_verified    host name neighbourhood group  \
0       80014485718            unconfirmed     Madaline            Brooklyn   
1       52335

C:\Users\tf\AppData\Local\Temp\ipykernel_14452\464628878.py:4: DtypeWarning: Columns (25) have mixed types. Specify dtype option on import or set low_memory=False.
  dataset_cont = pd.read_csv('C:/Users/tf/Desktop/airbnb_open_data.csv')


In [2]:

##### 1. Understanding the Dataset #####
#---------------------------------------

# We can get a good looking on the dataset with the next code below, where we get:
# The number of columns and records through the .info() statement
# A full report on the count of non null values for each column
# Additionally, the info statement display the data types in each column


In [3]:
print(f"Information about our selected dataset:\n {dataset_cont.info}")
# we can observe multiple columns with their non null counts and types (that have names needs standaized or unified later, Nan values, ..)

print(f"The length of the dataset:\n {len(dataset_cont)}")
# the total number of records is 102599

print(f"The columns we have:\n {dataset_cont.columns.tolist()}\n")
# to take a look on columns (maybe we will remove some of them, while formating others in a certain way)

Information about our selected dataset:
 <bound method DataFrame.info of              id                                              NAME  \
0       1001254                Clean & quiet apt home by the park   
1       1002102                             Skylit Midtown Castle   
2       1002403               THE VILLAGE OF HARLEM....NEW YORK !   
3       1002755                                               NaN   
4       1003689  Entire Apt: Spacious Studio/Loft by central park   
...         ...                                               ...   
102594  6092437                        Spare room in Williamsburg   
102595  6092990                     Best Location near Columbia U   
102596  6093542                    Comfy, bright room in Brooklyn   
102597  6094094                  Big Studio-One Stop from Midtown   
102598  6094647                              585 sf Luxury Studio   

            host id host_identity_verified    host name neighbourhood group  \
0       80014485718

In [4]:

##### 2. Cleaning and Preprocessing #####
#----------------------------------------
# This stage will have:
# Duplicated columns and rows dropped
# Unnecessary columns dropped too
# Date columns as objects transformed to DateTime
# $ signs removed from certain columns and the columns transformed to numerics
# Missing values handled according to the missings percentage
# Outliers removed
# Data prepared to be analysed in the next stage


In [5]:

# 1. Columns to drop (not useful for analysis or duplicated)
#-----------------------------------------------------------

prepro_dataset = dataset_cont # created a copy to save the data from the previous level

columns_drop = ['country', 'country code', 'Construction year', 'host_identity_verified']

# for example, the country and country_code are similar, and the dataset only discuss US data, so they are not useful
prepro_dataset = prepro_dataset.drop(columns_drop, axis = 1)
print(f"Remained columns: {prepro_dataset.columns.tolist()}")

Remained columns: ['id', 'NAME', 'host id', 'host name', 'neighbourhood group', 'neighbourhood', 'lat', 'long', 'instant_bookable', 'cancellation_policy', 'room type', 'price', 'service fee', 'minimum nights', 'number of reviews', 'last review', 'reviews per month', 'review rate number', 'calculated host listings count', 'availability 365', 'house_rules', 'license']


In [6]:

# 2. Duplicated rows to delete
print(f"Total duplicated rows: {prepro_dataset.duplicated().sum()}")

prepro_dataset = prepro_dataset.drop_duplicates(subset = ['id'])

#to check if there is any identical rows
perfect_duplicates = prepro_dataset.duplicated().sum()
print(f"Duplicated rows now: {perfect_duplicates}")

Total duplicated rows: 541
Duplicated rows now: 0


In [7]:

# 3. Handling Date Converting and Null Values
# -------------------------------------------
prepro_dataset = prepro_dataset.dropna(subset = ['number of reviews'])

#converting last review column to date
prepro_dataset['last review'] = pd.to_datetime(prepro_dataset['last review'], errors = 'coerce')

#filling missing reviews per month with 0
prepro_dataset['reviews per month'] = prepro_dataset['reviews per month'].fillna(0)

#filling missing ratings with median (as we will do with the numeric columns)
prepro_dataset['review rate number'] = prepro_dataset['review rate number'].fillna(prepro_dataset['review rate number'].median())

no_reviews = (prepro_dataset['number of reviews'] == 0) #this column has no nulls, the percentage was less than 5 so we removed the rows where missings occur
has_reviews = (prepro_dataset['number of reviews'] > 0) & (prepro_dataset['last review'].isna())

print(f"Listings number with no reviews: {no_reviews.sum()}")
print(f"Listings number with reviews but missing date: {has_reviews.sum()}")

#the rows that have no number of reviews should directly be replaced with the oldest value
prepro_dataset.loc[no_reviews, 'reviews per month'] = 0 #so the data is compatible betweenthe 3 variables we have about reviews in our dataset
prepro_dataset.loc[no_reviews, 'last review'] = prepro_dataset['last review'].min()

#handling missing according to the percentage of missing (dropping row or filling with oldest date)
missing_with_reviews = has_reviews.sum()
missing_percent = (missing_with_reviews / len(prepro_dataset)) * 100

if (missing_percent < 5):
    #dropping rows beacause it represents a very small sample of data
    prepro_dataset = prepro_dataset.drop(prepro_dataset[has_reviews].index)
    print(f"Deleted {missing_with_reviews} rows!")
else:
    #filling with the oldest date existed in the dataset
    oldest_date = prepro_dataset[prepro_dataset['last_review'].notnull()]['last_review'].min()
    prepro_dataset['last_review'] = prepro_dataset['last_review'].fillnull(oldest_date)
    print(f"Replaced {missing_with_reviews} rows with the oldest date existed {oldest_date.date()}")

#checking for impossible pairs in the dataset (like number or reviews is 0, but the reviews per month > 0)
inconsistent = prepro_dataset[
    ((prepro_dataset['number of reviews'] == 0) & (prepro_dataset['reviews per month'] > 0)) |
    ((prepro_dataset['number of reviews'] > 0) & (prepro_dataset['reviews per month'] == 0) & (prepro_dataset['last review'].notna()))
]
print(f"Inconsistent rows: {len(inconsistent)}")
prepro_dataset = prepro_dataset.drop(inconsistent.index)

Listings number with no reviews: 15673
Listings number with reviews but missing date: 35
Deleted 35 rows!
Inconsistent rows: 13


In [8]:
# 4. Removing $ and converting to number
# --------------------------------------
prepro_dataset['price'] = prepro_dataset['price'].replace('[\$,]', '', regex = True).astype(float)

prepro_dataset['service fee'] = prepro_dataset['service fee'].replace('[\$,]', '', regex = True).astype(float)


In [9]:

# 5. Splitting columns to 2 categories for better cleaning
# --------------------------------------------------------
# separating the columns that have objects of those who have numbers
object_cols = prepro_dataset.select_dtypes(include = ['object']).columns.tolist()
numeric_cols = prepro_dataset.select_dtypes(include = ['number']).columns.tolist()

# counting the columns of each type for future processing
print(f'Total count of objects columns: {len(object_cols)}, and the columns are: {object_cols}')
print(f'Total count of numbers columns: {len(numeric_cols)}, and the columns are: {numeric_cols}')




Total count of objects columns: 9, and the columns are: ['NAME', 'host name', 'neighbourhood group', 'neighbourhood', 'instant_bookable', 'cancellation_policy', 'room type', 'house_rules', 'license']
Total count of numbers columns: 12, and the columns are: ['id', 'host id', 'lat', 'long', 'price', 'service fee', 'minimum nights', 'number of reviews', 'reviews per month', 'review rate number', 'calculated host listings count', 'availability 365']


In [10]:
# 6. Handling Missing Values in Object Columns
# --------------------------------------------

obj_missing_info = []
action = 0
for col in object_cols:
    null_values = prepro_dataset[col].isnull().sum() #counting the null values
    empty_values = (prepro_dataset[col].astype(str).str.strip() == '').sum() #counting empty strings
    total_missing_percent = (null_values + empty_values) / len(prepro_dataset) * 100

    if (total_missing_percent < 5):
        prepro_dataset = prepro_dataset[prepro_dataset[col].notna()] #getting only not null values
        prepro_dataset = prepro_dataset[prepro_dataset[col].astype(str).str.strip() != ''] #getting the data with no empty strings
        action = "Fewer than 5%, rows where missed values occured is being removed."

    elif (5 < total_missing_percent < 30):
        prepro_dataset[col] = prepro_dataset[col].fillna(prepro_dataset[col].mode()) #filling nulls with the mode value of the column
        prepro_dataset[col] = np.where(prepro_dataset[col].astype(str).str.strip() == '', prepro_dataset[col].mode(), prepro_dataset[col]) #filling empty strings with the mode value
        action = "Over than 5% but less than 30%, rows are filled with the mode value."

    else:
        prepro_dataset = prepro_dataset.drop(col, axis = 1)
        action = "Over 30%, irreplacable missing, dropping the column."

    obj_missing_info.append({
        'Column': col,
        'Missing percent': round(total_missing_percent, 2),
        'Action we took': action
    })

missing_obj_df = pd.DataFrame(obj_missing_info)
print(missing_obj_df.to_string(index = False))

# a conclusion derived from the result table is that we need to remove the two columns: house_rules and license entirely
# because of the massive missing values percentage, while other columns can be filled normally with mode value during preprocessing



             Column  Missing percent                                                    Action we took
               NAME             0.24 Fewer than 5%, rows where missed values occured is being removed.
          host name             0.39 Fewer than 5%, rows where missed values occured is being removed.
neighbourhood group             0.02 Fewer than 5%, rows where missed values occured is being removed.
      neighbourhood             0.02 Fewer than 5%, rows where missed values occured is being removed.
   instant_bookable             0.09 Fewer than 5%, rows where missed values occured is being removed.
cancellation_policy             0.00 Fewer than 5%, rows where missed values occured is being removed.
          room type             0.00 Fewer than 5%, rows where missed values occured is being removed.
        house_rules            50.87              Over 30%, irreplacable missing, dropping the column.
            license           100.00              Over 30%, irreplacable 

In [11]:
# 7. Handling Missing Values in Number Columns
# --------------------------------------------

num_missing_info = []
action = 0

for col in numeric_cols:
    null_count = prepro_dataset[col].isnull().sum()
    missing_percent = (null_count / len(prepro_dataset)) * 100

    if (missing_percent < 5):
        prepro_dataset = prepro_dataset.dropna(subset = [col])
        action = "Fewer than 5%, rows where missed values occured is being removed."

    elif (missing_percent <= 30):
        prepro_dataset[col] = prepro_dataset[col].fillna(prepro_dataset[col].median())
        action = "Over than 5% but less than 30%, rows are filled with the mode value."

    else:
        prepro_dataset = prepro_dataset.drop(col, axis = 1)
        action = "Over 30%, irreplacable missing, dropping the column."

    num_missing_info.append({
        'Column': col,
        'Missing percent': round(missing_percent, 2),
        'Action we took': action
    })

missing_num_df = pd.DataFrame(num_missing_info)
print(missing_num_df.to_string(index = False))


                        Column  Missing percent                                                    Action we took
                            id             0.00 Fewer than 5%, rows where missed values occured is being removed.
                       host id             0.00 Fewer than 5%, rows where missed values occured is being removed.
                           lat             0.01 Fewer than 5%, rows where missed values occured is being removed.
                          long             0.00 Fewer than 5%, rows where missed values occured is being removed.
                         price             0.23 Fewer than 5%, rows where missed values occured is being removed.
                   service fee             0.23 Fewer than 5%, rows where missed values occured is being removed.
                minimum nights             0.37 Fewer than 5%, rows where missed values occured is being removed.
             number of reviews             0.00 Fewer than 5%, rows where missed values 

In [12]:

# 7. Removing spaces from all column names (standardizing)
prepro_dataset.columns = prepro_dataset.columns.str.replace(' ', '_')

#handling the uppercased NAME column
prepro_dataset = prepro_dataset.rename(columns = {'NAME': 'name'})

print(prepro_dataset.columns.tolist())
print(prepro_dataset.info())
# Finally saving the cleaned data
prepro_dataset.to_csv('C:/Users/tf/Desktop/airbnb_data_cleaned.csv', index = False, float_format = '%.6f')

['id', 'name', 'host_id', 'host_name', 'neighbourhood_group', 'neighbourhood', 'lat', 'long', 'instant_bookable', 'cancellation_policy', 'room_type', 'price', 'service_fee', 'minimum_nights', 'number_of_reviews', 'last_review', 'reviews_per_month', 'review_rate_number', 'calculated_host_listings_count', 'availability_365']
<class 'pandas.core.frame.DataFrame'>
Index: 99562 entries, 0 to 102040
Data columns (total 20 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   id                              99562 non-null  int64         
 1   name                            99562 non-null  object        
 2   host_id                         99562 non-null  int64         
 3   host_name                       99562 non-null  object        
 4   neighbourhood_group             99562 non-null  object        
 5   neighbourhood                   99562 non-null  object        
 6   lat                  